In [1]:
import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings('ignore')
import os

path = r'C:\Users\Anirudh\Desktop\olist_project\\'
os.chdir(path)

master = pd.read_csv('olist_master.csv')

# Create a SQLite database (creates a file on disk)
conn = sqlite3.connect('olist.db')

# Push the master dataframe into a SQL table
master.to_sql('orders', conn, if_exists='replace', index=False)

print("Database created ✅")
print("Table 'orders' has", len(master), "rows")

Database created ✅
Table 'orders' has 110173 rows


In [2]:
def run_query(query):
    return pd.read_sql_query(query, conn)

# Test it
test = run_query("SELECT * FROM orders LIMIT 5")
print(test)

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08

In [3]:
columns_query = run_query("PRAGMA table_info(orders)")
print(columns_query[['name', 'type']])

                             name     type
0                        order_id     TEXT
1                     customer_id     TEXT
2                    order_status     TEXT
3        order_purchase_timestamp     TEXT
4               order_approved_at     TEXT
5    order_delivered_carrier_date     TEXT
6   order_delivered_customer_date     TEXT
7   order_estimated_delivery_date     TEXT
8            actual_delivery_days  INTEGER
9         estimated_delivery_days  INTEGER
10                     delay_days  INTEGER
11                        is_late  INTEGER
12                  purchase_year  INTEGER
13                 purchase_month  INTEGER
14                   purchase_day     TEXT
15             customer_unique_id     TEXT
16       customer_zip_code_prefix  INTEGER
17                  customer_city     TEXT
18                 customer_state     TEXT
19                  order_item_id  INTEGER
20                     product_id     TEXT
21                      seller_id     TEXT
22         

In [4]:
q1 = """
SELECT 
    COUNT(*) as total_orders,
    SUM(is_late) as late_orders,
    ROUND(SUM(is_late) * 100.0 / COUNT(*), 2) as breach_rate_pct,
    ROUND(AVG(delay_days), 2) as avg_delay_days
FROM orders
"""
print(run_query(q1))

   total_orders  late_orders  breach_rate_pct  avg_delay_days
0        110173         8145             7.39          -11.43


In [5]:
q2 = """
SELECT 
    purchase_year,
    COUNT(*) as total_orders,
    SUM(is_late) as late_orders,
    ROUND(SUM(is_late) * 100.0 / COUNT(*), 2) as breach_rate_pct
FROM orders
GROUP BY purchase_year
ORDER BY purchase_year
"""
print(run_query(q2))

   purchase_year  total_orders  late_orders  breach_rate_pct
0           2016           317            5             1.58
1           2017         49538         3010             6.08
2           2018         60318         5130             8.50


In [6]:
q3 = """
SELECT 
    customer_state,
    COUNT(*) as total_orders,
    SUM(is_late) as late_orders,
    ROUND(SUM(is_late) * 100.0 / COUNT(*), 2) as breach_rate_pct,
    ROUND(AVG(delay_days), 2) as avg_delay_days
FROM orders
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY breach_rate_pct DESC
LIMIT 10
"""
print(run_query(q3))

  customer_state  total_orders  late_orders  breach_rate_pct  avg_delay_days
0             AL           427           99            23.19           -8.19
1             MA           799          157            19.65           -9.27
2             SE           375           61            16.27           -9.44
3             CE          1425          211            14.81          -10.48
4             PI           523           77            14.72          -10.91
5             BA          3683          475            12.90          -10.40
6             RJ         14140         1756            12.42          -11.39
7             PA          1054          128            12.14          -13.65
8             TO           310           37            11.94          -11.73
9             ES          2225          262            11.78          -10.04


In [7]:
q4 = """
SELECT 
    seller_id,
    seller_state,
    COUNT(*) as total_orders,
    SUM(is_late) as late_orders,
    ROUND(SUM(is_late) * 100.0 / COUNT(*), 2) as breach_rate_pct,
    ROUND(AVG(review_score), 2) as avg_review
FROM orders
GROUP BY seller_id, seller_state
HAVING COUNT(*) >= 20
ORDER BY breach_rate_pct DESC
LIMIT 10
"""
print(run_query(q4))

                          seller_id seller_state  total_orders  late_orders  \
0  2709af9587499e95e803a6498a5a56e9           SP            46           23   
1  821fb029fc6e495ca4f08a35d51e53a5           SP            25            9   
2  ede0c03645598cdfc63ca8237acbe73d           SP            47           16   
3  f76a3b1349b6df1ee875d1f3fa4340f0           SP            24            8   
4  54965bbe3e4f07ae045b90b0b8541f52           PR            81           26   
5  ad781527c93d00d89a11eecd9dcad7c1           SP            38           12   
6  7f152321c60a266edc53af1925ef96c1           SC            20            6   
7  2a1348e9addc1af5aaa619b1a3679d6b           MG            51           15   
8  4e5725ba188db8252977a4f0227bd462           SP            24            7   
9  835f0f7810c76831d6c7d24c7a646d4d           SP            48           14   

   breach_rate_pct  avg_review  
0            50.00        2.60  
1            36.00        3.50  
2            34.04        3.67 

In [8]:
q5 = """
SELECT 
    product_category_name_english as category,
    COUNT(*) as total_orders,
    ROUND(SUM(is_late) * 100.0 / COUNT(*), 2) as breach_rate_pct,
    ROUND(AVG(review_score), 2) as avg_review,
    ROUND(AVG(price), 2) as avg_price
FROM orders
WHERE product_category_name_english IS NOT NULL
GROUP BY category
HAVING COUNT(*) >= 50
ORDER BY breach_rate_pct DESC
LIMIT 10
"""
print(run_query(q5))

                  category  total_orders  breach_rate_pct  avg_review  \
0                    audio           362            12.15        3.84   
1       christmas_supplies           150            12.00        4.07   
2  fashion_underwear_beach           127            11.81        4.05   
3             home_confort           429            10.02        3.86   
4              electronics          2729             8.79        4.07   
5          books_technical           263             8.75        4.39   
6                     food           499             8.62        4.26   
7            health_beauty          9465             8.52        4.19   
8         office_furniture          1668             8.51        3.52   
9      musical_instruments           651             8.29        4.22   

   avg_price  
0     139.70  
1      58.25  
2      73.28  
3     135.22  
4      56.81  
5      71.11  
6      57.58  
7     130.28  
8     160.76  
9     283.13  


In [9]:
q6 = """
SELECT 
    customer_state,
    ROUND(SUM(price), 2) as total_revenue,
    ROUND(SUM(CASE WHEN is_late = 1 THEN price ELSE 0 END), 2) as revenue_at_risk,
    ROUND(SUM(CASE WHEN is_late = 1 THEN price ELSE 0 END) * 100.0 / SUM(price), 2) as pct_at_risk
FROM orders
GROUP BY customer_state
ORDER BY revenue_at_risk DESC
LIMIT 10
"""
print(run_query(q6))

  customer_state  total_revenue  revenue_at_risk  pct_at_risk
0             SP     5065805.31        312959.08         6.18
1             RJ     1759473.14        228594.82        12.99
2             MG     1552241.03         86495.31         5.57
3             BA      493584.14         66265.13        13.43
4             RS      728205.48         49863.27         6.85
5             SC      507012.13         42522.29         8.39
6             ES      268643.45         36731.23        13.67
7             PR      666063.51         33235.55         4.99
8             CE      219677.39         31874.71        14.51
9             PE      251889.49         28267.77        11.22


In [12]:
q8 = """
SELECT 
    seller_state,
    customer_state,
    COUNT(*) as total_orders,
    ROUND(AVG(delay_days), 2) as avg_delay_days,
    ROUND(SUM(is_late) * 100.0 / COUNT(*), 2) as breach_rate_pct
FROM orders
GROUP BY seller_state, customer_state
HAVING COUNT(*) >= 50
ORDER BY breach_rate_pct DESC
LIMIT 10
"""
print(run_query(q8))

  seller_state customer_state  total_orders  avg_delay_days  breach_rate_pct
0           MA             SP           130           -8.89            25.38
1           RJ             CE            56           -4.57            25.00
2           SP             AL           273           -7.70            23.81
3           SP             MA           548           -8.91            21.53
4           MG             PB            58          -12.64            18.97
5           SP             SE           237           -8.78            18.14
6           MG             MA            67          -10.12            17.91
7           SP             PI           362          -10.05            17.40
8           MG             PA            92          -10.04            17.39
9           SP             CE          1090          -10.55            14.95


In [13]:
q9 = """
SELECT 
    purchase_year,
    purchase_month,
    COUNT(*) as total_orders,
    ROUND(SUM(is_late) * 100.0 / COUNT(*), 2) as breach_rate_pct
FROM orders
WHERE purchase_year IN (2017, 2018)
GROUP BY purchase_year, purchase_month
ORDER BY purchase_year, purchase_month
"""
print(run_query(q9))

    purchase_year  purchase_month  total_orders  breach_rate_pct
0            2017               1           911             2.85
1            2017               2          1845             3.20
2            2017               3          2897             5.11
3            2017               4          2569             6.85
4            2017               5          4003             3.32
5            2017               6          3489             3.27
6            2017               7          4416             3.37
7            2017               8          4797             3.00
8            2017               9          4736             4.88
9            2017              10          5214             4.74
10           2017              11          8474            12.95
11           2017              12          6187             7.86
12           2018               1          8037             6.23
13           2018               2          7518            15.16
14           2018        

In [14]:
q10 = """
SELECT 
    purchase_day,
    COUNT(*) as total_orders,
    ROUND(SUM(is_late) * 100.0 / COUNT(*), 2) as breach_rate_pct
FROM orders
GROUP BY purchase_day
ORDER BY breach_rate_pct DESC
"""
print(run_query(q10))

  purchase_day  total_orders  breach_rate_pct
0       Monday         17973             8.06
1      Tuesday         17857             7.80
2       Friday         15693             7.63
3    Wednesday         17217             7.12
4       Sunday         13126             6.97
5     Thursday         16431             6.95
6     Saturday         11876             6.92


In [15]:
# Save your top findings as CSVs for the Power BI dashboard later
run_query(q3).to_csv('sql_top_states_breach.csv', index=False)
run_query(q4).to_csv('sql_worst_sellers.csv', index=False)
run_query(q5).to_csv('sql_category_performance.csv', index=False)
run_query(q6).to_csv('sql_revenue_at_risk.csv', index=False)

print("Key query results saved ✅")

Key query results saved ✅


In [16]:
# ===================== DAY 6 OBSERVATIONS =====================
# 1. Did breach rate improve from 2017 to 2018?       = yes
# 2. Worst seller-to-customer state route             = sp
# 3. Worst day of week to place an order              = Monday
# 4. Any seasonal pattern in breach rate (Q9)         = High breach rate in april and november probaly because of more busy time due to easter and chritmas.
# 5. One new insight not found in Python EDA          = Breah rate has seasonal pattern
# ==============================================================